In [10]:
# -----------------------------------------------------
# Imports
# -----------------------------------------------------
from dash import Dash, dcc, html, dash_table, jupyter_dash
from dash.dependencies import Input, Output
import requests
import pandas as pd

### Configurations & API Setup

---

**Explanation:**  
- `BASE_URL` → Base address of PyPI API server. Changing this will point all requests to a different server.  

- **API Endpoints:**  
  - `TOTAL_PACKAGES_ENDPOINT` → Returns the total number of PyPI packages.  
  - `PACKAGE_VERSIONS_ENDPOINT` → Returns versions of each package.  
  - `RELEASES_OVER_YEARS_ENDPOINT` → Returns the number of package releases per year.  
  - `ALL_SOURCES_COVERAGE_ENDPOINT` → Returns coverage statistics from multiple sources.  
  - `PYPI_DETAILS_ENDPOINT` → Returns detailed metadata for each PyPI project.  

- `get_json(endpoint)` → A helper function to simplify API calls:  
  1. Makes a GET request to the specified endpoint.  
  2. Raises an error if the request fails.  
  3. Returns the response as JSON for further processing.

---

**Data Exploration:**  
- You can test this by running, for example:  
  ```python
  get_json(TOTAL_PACKAGES_ENDPOINT)


In [11]:
# -----------------------------------------------------
# Configurations
# -----------------------------------------------------
BASE_URL = "http://127.0.0.1:8080/pypi"

# Endpoints
TOTAL_PACKAGES_ENDPOINT = f"{BASE_URL}/total-packages"
PACKAGE_VERSIONS_ENDPOINT = f"{BASE_URL}/package-versions"
RELEASES_OVER_YEARS_ENDPOINT = f"{BASE_URL}/releases-over-years"
ALL_SOURCES_COVERAGE_ENDPOINT = f"{BASE_URL}/all_sources_coverage"
PYPI_DETAILS_ENDPOINT = f"{BASE_URL}/project_details"

# Function to get JSON from API
def get_json(endpoint):
    resp = requests.get(endpoint)
    resp.raise_for_status()
    return resp.json()

### Loading PyPI Data (Total Packages & Project Details)
This cell fetches data from the API and organizes it into a clean **Pandas DataFrame** for analysis and visualization.

### Fetch Total Packages
- Calls `get_json(TOTAL_PACKAGES_ENDPOINT)`.
- Extracts the value of `"total_packages"` and stores it in the variable `total_packages`.  
- This gives the overall count of packages available in the PyPI dataset which are related to GA4GH.
- Our search is based on two keywords: 1. GA4GH, 2. Global Alliance for Genomics and Health

### Build dataframe
- Loops through each package and extracts relevant fields into rows and finally transforming it into dataframe.

### Data Exploration
- df.head() -> Preview the first 5 rows
- df.shape -> Check number of rows and columns
- df.info() -> Get column data types and non-null counts

In [12]:
# -----------------------------------------------------
# 1. Total packages
# -----------------------------------------------------
total_packages_resp = get_json(TOTAL_PACKAGES_ENDPOINT)
total_packages = int(total_packages_resp.get("total_packages", 0))

# -----------------------------------------------------
# 2. Project details
# -----------------------------------------------------
details_data = get_json(PYPI_DETAILS_ENDPOINT)

# Build DataFrame
rows = []
for pkg in details_data:
    versions_list = pkg.get("versions", [])
    versions_count = versions_list[0].get("versions", 0) if versions_list else 0
    rows.append({
        "project_name": pkg.get("project_name"),
        "description": pkg.get("description"),
        "author_name": pkg.get("author_name"),
        "author_email": pkg.get("author_email"),
        "category": pkg.get("category"),
        "versions_count": versions_count,
    })

df = pd.DataFrame(rows)

### Exploring the DataFrame
- Now that we have loaded our PyPI project details into a DataFrame (`df`), let’s **explore the dataset** to better understand its structure and contents.
- This step is helpful for both technical and non-technical users; it provides an overview of what kind of information is stored and how big the dataset is.

In [13]:
# Preview the first 5 rows
display(df.head())

# Shape of the dataset (rows, columns)
print("Shape of DataFrame:", df.shape)

# Info about columns and datatypes
print("\nDataFrame Info:")
print(df.info())

,project_name,description,author_name,author_email,category,versions_count
0,data-repo-client,Data Repository API,OpenAPI Generator community,team@openapitools.org,Implementation,627
1,cwltool,Common workflow language reference implementation,Common workflow language working group,common-workflow-language@googlegroups.com,Implementation,436
2,pyphetools,Generate and work with GA4GH phenopackets,None,Peter Robinson <peter.robinson@bih-charite.de>...,Implementation,199
3,planemo,Command-line utilities to assist in building t...,None,Galaxy Project and Community <galaxy-committer...,Implementation,148
4,bento-lib,A set of common utilities and helpers for Bent...,David Lougheed,david.lougheed@mail.mcgill.ca,Reference,105


Shape of DataFrame: (82, 6)

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82 entries, 0 to 81
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   project_name    82 non-null     object
 1   description     81 non-null     object
 2   author_name     63 non-null     object
 3   author_email    75 non-null     object
 4   category        82 non-null     object
 5   versions_count  82 non-null     int64 
dtypes: int64(1), object(5)
memory usage: 4.0+ KB
None


### Dynamic DataTable for High-Level View
This section creates an **interactive dashboard** using Dash.  
It allows users to explore the PyPI dataset in a table format with search, sorting, and filtering features.

### Key Features:
- **Total Packages Display** -> Shows the total number of PyPI packages available.  
- **Search Box** -> Lets you search across all columns as you type (keystroke search).  
- **DataTable** -> An interactive table where you can:
  - Navigate through pages (10 rows per page by default).
  - Sort columns by clicking headers.
  - Filter each column individually using Dash’s native filter.

### Exploration Opportunities for Non-Technical Users:
- **Change the number of rows shown per page** -> by modifying `page_size=10`.  
- **Adjust search behavior** -> switching `debounce=False` to `debounce=True` will make the search run only after pressing Enter.
- **Style Customization** -> If you want to play with Colors, fonts, and table borders it can be easily adjusted in the `style_*` sections.  

This DataTable is meant to give users a **high-level overview** of the dataset and quickly locate packages of interest.


In [14]:
# -----------------------------------------------------
# 3. Dynamic datatable for high level view
# -----------------------------------------------------
app = Dash(__name__)

app.layout = html.Div([
    html.H2(
        f"Total PyPI Packages: {total_packages}",
        style={'textAlign': 'center', 'margin-bottom': '20px', 'color': '#2C3E50'}
    ),

    # Search box for global search - filters across all columns
    dcc.Input(
        id='table-search',
        type='text',
        placeholder='Search projects...',
        debounce=False,   # keystroke-based search (change to True for enter-based search)
        style={
            'margin-bottom': '15px',
            'width': '350px',
            'padding': '8px',
            'border-radius': '5px',
            'border': '1px solid #ccc'
        }
    ),

    # Interactive DataTable
    dash_table.DataTable(
        id='projects-table',
        columns=[{"name": i, "id": i} for i in df.reset_index().columns],
        
        data=df.reset_index().to_dict('records'),
        page_size=10,
        sort_action='native', # enable sorting by column
        filter_action='native',  # native per-column filter
        
        # Styling options
        style_table={'overflowX': 'auto', 'border': '1px solid #ccc', 'border-radius': '5px'},
        style_header={'backgroundColor': '#34495E', 'color': 'white', 'fontWeight': 'bold'},
        style_cell={
            'textAlign': 'left',
            'padding': '8px',
            'minWidth': '100px', 'width': '150px', 'maxWidth': '250px',
            'whiteSpace': 'nowrap',
            'overflow': 'hidden',
            'textOverflow': 'ellipsis',
            'border-bottom': '1px solid #ddd'
        },
        style_data_conditional=[{'if': {'row_index': 'odd'}, 'backgroundColor': '#f9f9f9'}]
    ),
])

# -----------------------------------------------------
# Callback: Update table dynamically based on search box
# -----------------------------------------------------
@app.callback(
    Output('projects-table', 'data'),
    Input('table-search', 'value')
)
def update_table(search_value):
    """
    Filters the DataFrame rows based on the search input.
    Runs on every keystroke unless debounce=True.
    """
    if not search_value:
        return df.reset_index().to_dict('records')
    filtered = df.reset_index()[df.reset_index().apply(
        lambda row: row.astype(str).str.contains(search_value, case=False).any(), axis=1
    )]
    return filtered.to_dict('records')

### Dynamic Bar Graph from DataTable

This section creates a **bar chart** that dynamically updates based on the DataTable above.  
The chart shows the **number of versions for each package** currently displayed in the table.

### Key Features:
- **Linked to DataTable** -> The graph only uses the data from the visible rows (current page).  
- **Updates automatically** when:
  - You filter or search in the table.
  - You change pages.  
- **Hover Tooltips** -> Show extra information such as project name, category, and version count when hovering over the bar.  
- **Custom Styling** -> Titles, colors, and hover formatting.

### Exploration Opportunities for Non-Technical Users:
- **Change the chart type** -> Replace `"type": "bar"` with `"type": "scatter"` for a line or `"pie"` for a pie chart.  
- **Adjust hover text** -> Customize what appears on hover by editing the `hover_texts` list.  
- **Change colors** -> Update `"marker": {"color": "#2E86C1"}` to another hex code (e.g., `"#E74C3C"` for red).
- **Modify axis titles** -> In `xaxis` and `yaxis`, replace `"project_name"` or `"versions_count"` with friendlier labels.

This visualization gives a **clear snapshot of version counts** for the subset of packages currently being inspected in the table.


In [15]:
# -----------------------------------------------------
# 4. Dynamic bar graph inheriting data from datatable
# -----------------------------------------------------

app.layout.children.append(
    dcc.Graph(id="datatable-bar")
)

# -----------------------------------------------------
# Callback: Update Bar Graph based on DataTable state
# -----------------------------------------------------
@app.callback(
    Output("datatable-bar", "figure"),
    Input("projects-table", "derived_virtual_data"),    # table rows after filtering
    Input("projects-table", "page_current"),            # current page index
    Input("projects-table", "page_size")                # rows per page
)

def update_bar(rows, page_current, page_size):
    # If DataTable rows exist, use them; else fallback to original DataFrame
    dff = pd.DataFrame(rows) if rows else df.copy()

    # If "project_name" missing, rename index
    if "project_name" not in dff.columns:
        if "index" in dff.columns:
            dff = dff.rename(columns={"index": "project_name"})

    # Provide safe defaults for paging
    page_current = page_current or 0
    page_size = page_size or 10

    # Slice the data for the current page
    start = page_current * page_size
    end = start + page_size
    page_data = dff.iloc[start:end]

    # Create hover text: shorten long description
    hover_texts = [
        f"<b>{row['project_name']}</b><br>"
        f"Category: {row.get('category', '')}<br>"
        f"Versions: {str(row.get('versions_count', ''))}"
        for _, row in page_data.iterrows()
    ]

    fig = {
        "data": [
            {
                "x": page_data["project_name"],      # X-axis: package names
                "y": page_data["versions_count"],    # Y-axis: version counts
                "type": "bar",                       # Bar chart
                "marker": {"color": "#2E86C1"},        # Bar color
                "hovertext": hover_texts,            # Custom hover info
                "hoverinfo": "text"                  # Only show hover text
            }
        ],
        "layout": {
            "title": {
                "text": "Package Versions Count (Current Page)",
                "x": 0.5,          # center title
                "xanchor": "center",
                "yanchor": "top",
                "font": {"size": 20, "color": "#2C3E50"}
            },
            "xaxis": {"title": "project_name", "tickangle": -45},
            "yaxis": {"title": "versions_count"},
            "plot_bgcolor": "#f9f9f9",
            "paper_bgcolor": "#ffffff",
            "margin": {"b": 120}
        }
    }
    return fig

### Category Distribution Graph
This section creates a **donut-style pie chart** that shows the distribution of packages across different categories.  

### Key Features:
- **Dynamic** -> The chart updates whenever the DataTable changes (through search, filters, or pagination).  
- **Counts categories** -> Calculates how many packages fall under each category and visualizes them proportionally.  
- **Hover Tooltips** -> Show category name, count, and percentage.  
- **Donut Style** -> A hollow pie chart with labels around the circle.

### Exploration Opportunities for Non-Technical Users:
- **Change chart type** -> Replace `"type": "pie"` with `"bar"` or `"funnel"` for alternative views.  
- **Make it a full pie chart** -> Remove `"hole": 0.4` if you prefer a solid pie instead of a donut.  
- **Customize labels** -> Update `"textinfo": "label+percent"` to `"label+value"` if you want raw counts instead of percentages.  
- **Colors** -> Dash automatically assigns colors, but you can add `"marker": {"colors": [...]}` with custom hex codes.  

This graph provides an **at-a-glance understanding** of how projects are distributed by category.


In [16]:
# -----------------------------------------------------
# 5. Category distribution
# -----------------------------------------------------
app.layout.children.append(
    dcc.Graph(id="category-distribution")
)

@app.callback(
    Output("category-distribution", "figure"),
    Input("projects-table", "derived_virtual_data")
)
def update_category_distribution(rows):
    dff = pd.DataFrame(rows) if rows is not None else df.reset_index()

    if "category" not in dff.columns or dff.empty:
        return {"data": [], "layout": {"title": "No category data available"}}

    # Count categories
    cat_counts = dff["category"].value_counts().reset_index()
    cat_counts.columns = ["category", "count"]

    return {
        "data": [{
            "labels": cat_counts["category"],
            "values": cat_counts["count"],
            "type": "pie",
            "hole": 0.4,
            "textinfo": "label+percent",
            "hoverinfo": "label+value+percent"
        }],
        "layout": {
            "title": {
                "text": "Category Distribution",
                "x": 0.5,          # center title
                "xanchor": "center",
                "yanchor": "top",
                "font": {"size": 20, "color": "#2C3E50"}
            },
            "plot_bgcolor": "#f9f9f9",
            "paper_bgcolor": "#ffffff"
        }
    }
    return fig

### Running the Dash App
The `app.run()` command starts the **Dash application** inside this notebook.  
At this stage, the app will render **all graphs and the DataTable one after another** (in a single scroll view).  

This happens because Dash is designed to use a **single `app.layout` container** that holds all the components (table + graphs).  
So whenever we add a new graph to the layout, it simply stacks it below the previous components.

### Important Notes
- In the future, we can make the dashboard more modular, e.g.:
  - Place each graph in its own section with tabs, collapsible panels, or navigation.  
  - Organize layouts into multiple pages (so each chart gets its own page).  
  - Provide non-technical users with easier controls (like dropdowns or sliders) to switch between visualizations.

For now, `app.run()` creates the **DataTable**, **bar chart**, and **category distribution chart** — are all interactive and visible together in the same interface.

In [17]:
app.run()

In [ ]:
# -----------------------------------------------------
# 6. Releases over years
# -----------------------------------------------------
app.layout.children.append(
    dcc.Graph(id="releases-over-years")
)

@app.callback(
    Output("releases-over-years", "figure"),
)
def releases_over_years(rows):
    releases = get_json(RELEASES_OVER_YEARS_ENDPOINT)
    dff = pd.DataFrame(releases)
    print(dff)
    return {
        "data": [
            {
                "x": dff["year"],
                "y": dff["releases"],
                "type": "bar",
                "marker": {"color": "#27AE60"},
                "hovertemplate": "%{x}: %{y} releases<extra></extra>"
            }
        ],
        "layout": {
            "title": {
                "text": "PyPI Releases Per Year",
                "x": 0.5,
                "xanchor": "center",
                "font": {"size": 18, "color": "#2C3E50"}
            },
            "xaxis": {"title": "Year"},
            "yaxis": {"title": "Number of Releases"},
            "plot_bgcolor": "#f9f9f9",
            "paper_bgcolor": "#ffffff"
        }
    }
    return fig

